In [1]:
!pip install -q pyspark

In [2]:
from google.colab import files
uploaded = files.upload()

Saving users.csv to users.csv
Saving cleaned_expenses.csv to cleaned_expenses.csv


In [3]:
from pyspark.sql import SparkSession, functions as F
spark = SparkSession.builder.appName("ExpenseETL").getOrCreate()
ALERT_THRESHOLD = 0.8   # alert if monthly spend is more than 80% of incom
expenses = spark.read.csv("cleaned_expenses.csv", header=True, inferSchema=True)
expenses = expenses.withColumn("month", F.date_format("month", "yyyy-MM"))
users = spark.read.csv("users.csv", header=True, inferSchema=True)
print("Expenses rows:", expenses.count())
print("Users rows:", users.count())

Expenses rows: 100000
Users rows: 200


In [4]:
combined = expenses.join(users, on="user_id", how="inner")
combined.show(5)

+-------+----------+---------+-------+---------------+-------+-------------+-----------------+--------------+
|user_id|      date| category| amount|    description|  month|         name|            email|monthly_income|
+-------+----------+---------+-------+---------------+-------+-------------+-----------------+--------------+
|      1|2025-01-01|   Health| 742.87|   Doctor visit|2025-01|Harish Pillai|user1@example.com|         83300|
|      3|2025-01-01|     Food| 137.95|         Snacks|2025-01|   Priya Iyer|user3@example.com|         81800|
|      4|2025-01-01|Utilities| 651.62|Mobile recharge|2025-01| Harish Kumar|user4@example.com|         78400|
|      5|2025-01-01| Shopping|1625.65|     Home items|2025-01|Karthik Menon|user5@example.com|         55400|
|      5|2025-01-01|     Food| 229.49|  Food delivery|2025-01|Karthik Menon|user5@example.com|         55400|
+-------+----------+---------+-------+---------------+-------+-------------+-----------------+--------------+
only showi

In [5]:
summary = (
    combined
    .groupBy("user_id", "name", "month", "monthly_income")
    .agg(F.round(F.sum("amount"), 2).alias("monthly_spend"))
    .withColumn("savings", F.round(F.col("monthly_income") - F.col("monthly_spend"), 2))
    .withColumn(
        "alert",
        F.when(F.col("savings") < 0, "ALERT: Overspent income")
         .when(F.col("monthly_spend") > ALERT_THRESHOLD * F.col("monthly_income"), "ALERT: Spend above 80% of income")
         .otherwise("OK")
    )
    .orderBy("user_id", "month")
)

summary.show(10)
summary.groupBy("alert").count().show(truncate=False)

+-------+-------------+-------+--------------+-------------+---------+--------------------+
|user_id|         name|  month|monthly_income|monthly_spend|  savings|               alert|
+-------+-------------+-------+--------------+-------------+---------+--------------------+
|      1|Harish Pillai|2025-01|         83300|     29138.94| 54161.06|                  OK|
|      1|Harish Pillai|2025-02|         83300|     20146.24| 63153.76|                  OK|
|      1|Harish Pillai|2025-03|         83300|     49750.02| 33549.98|                  OK|
|      1|Harish Pillai|2025-04|         83300|      43676.5|  39623.5|                  OK|
|      1|Harish Pillai|2025-05|         83300|     16111.01| 67188.99|                  OK|
|      1|Harish Pillai|2025-06|         83300|      43047.4|  40252.6|                  OK|
|      1|Harish Pillai|2025-07|         83300|     20583.83| 62716.17|                  OK|
|      1|Harish Pillai|2025-08|         83300|     97724.89|-14424.89|ALERT: Ove

In [6]:
summary.toPandas().to_csv("monthly_summary.csv", index=False)
files.download("monthly_summary.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>